# 05 Masked Self-Attention

## 목적
Q, K, V, scaled dot-product attention, lower-triangular causal mask를 사용해 single-head masked self-attention을 확인한다. 미래 위치의 attention probability가 0인지 직접 출력한다.

In [1]:
from pathlib import Path
import sys, torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.tokenizer import CharTokenizer, load_text
from src.baselines import SingleHeadMaskedSelfAttention

torch.manual_seed(42)
text = load_text(str(ROOT / 'data/korean_finance_corpus.txt'))
tokenizer = CharTokenizer(text)
print('sample text:', text[:30])

sample text: 인공지능은 금융 시장을 이해하는 새로운 도구가 될 수 


## 핵심 코드
Q는 현재 위치가 찾는 정보, K는 각 위치의 주소, V는 실제로 섞이는 값이다. `Q @ K.T / sqrt(head_size)`로 점수를 만들고 lower-triangular mask로 미래 위치를 가린다.

In [2]:
B, T, C, H = 1, 6, 12, 6
x = torch.randn(B, T, C)
attn = SingleHeadMaskedSelfAttention(n_embd=C, head_size=H, block_size=T)
out, weights = attn(x)
print('input shape:', tuple(x.shape))
print('attention weights shape:', tuple(weights.shape))
print('output shape:', tuple(out.shape))

input shape: (1, 6, 12)
attention weights shape: (1, 6, 6)
output shape: (1, 6, 6)


In [3]:
torch.set_printoptions(precision=3, sci_mode=False)
print(weights[0])
future_probs = torch.triu(weights[0], diagonal=1)
print('max future probability:', float(future_probs.max()))

tensor([[1.000, 0.000, 0.000, 0.000, 0.000, 0.000],
        [0.548, 0.452, 0.000, 0.000, 0.000, 0.000],
        [0.224, 0.644, 0.132, 0.000, 0.000, 0.000],
        [0.240, 0.259, 0.256, 0.245, 0.000, 0.000],
        [0.260, 0.215, 0.180, 0.175, 0.170, 0.000],
        [0.116, 0.181, 0.220, 0.117, 0.166, 0.201]], grad_fn=<SelectBackward0>)
max future probability: 0.0


/tmp/ipykernel_49409/1277605267.py:4: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  print('max future probability:', float(future_probs.max()))


## 이 단계의 한계
단일 head는 하나의 관점으로만 과거 정보를 읽는다. 또한 attention 뒤의 feed-forward, residual, LayerNorm, dropout, stacked block이 없다.

## 다음 단계가 해결하는 점
Notebook 06은 multi-head attention과 Transformer block을 쌓아 최종 Tiny GPT를 학습한다.